# Batch Deployment

Batch inference is the most common way of deploying machine learning models. This lesson introduces various strategies for deploying models using batch including Spark. In addition, we will show how to enable optimizations for Delta tables.

**Learning Objectives:**

*By the end of this demo, you will be able to:*

* Load a logged Model Registry model using `pyfunc`.
* Compute predictions using `pyfunc` APIs.
* Perform batch inference using Feature Engineering's `score_batch` method.
* Materialize predictions into inference tables (Delta Lake).
* Perform common write optimizations like liquid clustering, predictive optimization to maximize data skipping and on inference tables.

# Requirements

Please review the following requirements before starting the lesson:
* To run this notebook, you need to use one of the following Databricks runtime(s): **{{supported_dbrs}}**

🚨 **Prerequisites:**

* **Feature Engineering** and **Feature Store** are not focus of this lesson. This course expect that you already know these topics. If not, you can check the **Data Preparation for Machine Learning** course.
* Model development with MLflow is not in the scope of this course. If you need to refresh your knowledge about model tracking and logging, you can check the **Machine Learning Model Development** course.

https://www.kaggle.com/datasets/blastchar/telco-customer-churn

In [0]:
%pip install databricks-feature-engineering

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 28.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3ba8ee91-c061-43f6-9b5c-53ac320dc889
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-3ba8ee91-c061-43f6-9b5c-53ac320dc889
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os

# 1. Configurar as credenciais do Kaggle no ambiente do cluster
os.environ['KAGGLE_USERNAME'] = "fabienecasamento" #"SEU_USUARIO_AQUI"
with open('/Workspace/Users/fabieneaulas@gmail.com/ML_DEPLOY/chave_kaggle.txt', 'r') as leitura_chave:
    kaggle_key = leitura_chave.read().strip()
# Extrair os dados antes do ';|' na variável kaggle_key
kaggle_key_before_delimiter = kaggle_key.split(';')[0]


os.environ['KAGGLE_KEY'] = kaggle_key_before_delimiter #"SUA_CHAVE_AQUI" # token databricks

In [0]:
# 2. Instalar a biblioteca do Kaggle via comando de terminal no cluster
# (O caractere '!' permite rodar comandos shell diretamente do notebook)
!pip install -q kaggle

# 3. Criar uma pasta local temporária no driver e baixar o dataset do link informado
!mkdir -p /tmp/kaggle_data
!kaggle datasets download -d blastchar/telco-customer-churn -p /tmp/kaggle_data --unzip

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
100%|████████████████████████████████████████| 172k/172k [00:00<00:00, 3.77MB/s]



In [0]:
# 4. Copiar o arquivo do /tmp para o Workspace (serverless requer /Workspace ou Volumes)
import shutil
import os

workspace_dir = "/Workspace/Users/fabieneaulas@gmail.com/ML_DEPLOY/kaggle_data"
os.makedirs(workspace_dir, exist_ok=True)
shutil.copy("/tmp/kaggle_data/WA_Fn-UseC_-Telco-Customer-Churn.csv", f"{workspace_dir}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 5. Abrir e ler o arquivo utilizando o PySpark
# O arquivo baixado do Kaggle adota o nome padrão original da IBM ("WA_Fn-UseC_-...")
dataset_path = f"{workspace_dir}/WA_Fn-UseC_-Telco-Customer-Churn.csv"

telco_df = spark.read.csv(
    dataset_path, 
    inferSchema=True, 
    header=True, 
    multiLine=True, 
    escape='"'
)

# 6. Visualizar o DataFrame carregado com sucesso no Spark
display(telco_df.show(8))

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
display(telco_df.toPandas().tail(8))

customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
8456-QDAVC,Male,0,No,No,19,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),78.7,1495.1,No
7750-EYXWZ,Female,0,No,No,12,No,No phone service,DSL,No,Yes,Yes,Yes,Yes,Yes,One year,No,Electronic check,60.65,743.3,No
2569-WGERO,Female,0,No,No,72,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.4,No
6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.8,1990.5,No
2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.2,7362.9,No
4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.6,346.45,No
8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.4,306.6,Yes
3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No


In [0]:
telco_df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
from pyspark.sql.functions import col


# dataset path
#dataset_p_telco = f"{DA.paths.datasets}/telco/telco-customer-churn.csv"

# features to use
primary_key = "customerID"
response = "Churn"
features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"] # Keeping numerical only for simplicity and demo purposes

#telco_df = telco_df.withColumn("TotalCharges", col("TotalCharges").cast('double'))\
#    .withColumn("SeniorCitizen", col("SeniorCitizen").cast('double'))\
#    .withColumn("Tenure", col("tenure").cast('double'))\
#    .na.drop(how='any')

    
from pyspark.sql.functions import expr

telco_df = telco_df.withColumn("TotalCharges", expr("try_cast(TotalCharges as double)"))\
    .withColumn("SeniorCitizen", expr("try_cast(SeniorCitizen as double)"))\
    .withColumn("Tenure", expr("try_cast(tenure as double)"))\
    .na.drop(how='any')

# Split with 80 percent of the data in train_df and 20 percent of the data in test_df
train_df, test_df = telco_df.randomSplit([.8, .2], seed=42)

# Separate features and ground-truth
features_df = train_df.select(primary_key, *features)
response_df = train_df.select(primary_key, response)

# review the features dataset


In [0]:
telco_df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: double (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Tenure: double (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [0]:
display(features_df.show(3))

+----------+-------------+------+--------------+------------+
|customerID|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|
+----------+-------------+------+--------------+------------+
|0002-ORFBO|          0.0|   9.0|          65.6|       593.3|
|0003-MKNFE|          0.0|   9.0|          59.9|       542.4|
|0004-TLHLJ|          0.0|   4.0|          73.9|      280.85|
+----------+-------------+------+--------------+------------+
only showing top 3 rows


In [0]:
display(response_df.show(3))

+----------+-----+
|customerID|Churn|
+----------+-----+
|0002-ORFBO|   No|
|0003-MKNFE|   No|
|0004-TLHLJ|  Yes|
+----------+-----+
only showing top 3 rows


# Batch Deployment - Without Feature Store

This demo will cover two main batch deployment methods. The first method is deploying models without a feature table. For the second method, we will use a feature table to train the model and later use the feature table for inference.

## Setup Model Registry with UC

Before we start model deployment, we need to fit and register a model. In this demo, **we will log models to Unity Catalog**, which means first we need to setup the **MLflow Model Registry URI**.

In [0]:
import mlflow


# Point to UC model registry
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

# helper function that we will use for getting latest version of a model
def get_latest_model_version(model_name):
    """Helper function to get latest model version"""
    model_version_infos = client.search_model_versions(f"name = '%s'" % model_name)
    return max([model_version_info.version for model_version_info in model_version_infos])

In [0]:
# List available catalogs and schemas in Unity Catalog
catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
display(catalogs)



_1
samples
system
workspace


In [0]:
# Choose a catalog and list schemas
#selected_catalog = catalogs[2]  # or set manually
#print(selected_catalog)
#schemas = [row.schema for row in spark.sql(f"SHOW SCHEMAS IN {selected_catalog}").collect()]
#display(schemas)


In [0]:

# Set catalog_name and schema_name variables
catalog_name = catalogs[2] 
#schema_name = schemas[0]  # or set manually

schema_name = 'information_schema' # sem premissão
print(f"{catalog_name}.{schema_name}")

workspace.information_schema


In [0]:
# Train a sklearn Decision Tree Classification model
from sklearn.tree import DecisionTreeClassifier
from mlflow.models import infer_signature


# Set catalog_name and schema_name variables
catalog_name = catalogs[2] 
#schema_name = schemas[0]  # or set manually

schema_name = 'default'
print(f"{catalog_name}.{schema_name}")

# Covert data to pandas dataframes
X_train_pdf = features_df.drop(primary_key).toPandas()
Y_train_pdf = response_df.drop(primary_key).toPandas()
clf = DecisionTreeClassifier(max_depth=3, random_state=42)

# Use 3-level namespace for model name
model_name = f"{catalog_name}.{schema_name}.ml_model"

with mlflow.start_run(run_name="Model-Batch-Deployment-Demo") as mlflow_run:

    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        log_models=False,
        log_post_training_metrics=True,
        silent=True
    )

    clf.fit(X_train_pdf, Y_train_pdf)

    # Log model and push to registry
    signature = infer_signature(X_train_pdf, Y_train_pdf)
    mlflow.sklearn.log_model(
        clf,
        artifact_path="decision_tree",
        signature=signature,
        registered_model_name=model_name
    )

    # Set model alias (i.e. Baseline)
    client.set_registered_model_alias(model_name, "Baseline", get_latest_model_version(model_name))

workspace.default


2026/06/05 01:10:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/2770952798348874/models/m-7000149c905a45fdac2d95e68368d6b9?o=7474657872577658
Registered model 'workspace.default.ml_model' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.ml_model': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/ml_model/version/2?o=7474657872577658


![image_1780618385106.png](./image_1780618385106.png "image_1780618385106.png")

![image_1780618435965.png](./image_1780618435965.png "image_1780618435965.png")

In [0]:
#página 36/38
# clicando no model-batch-deployment-demo
# no mlflow

![image_1780618515646.png](./image_1780618515646.png "image_1780618515646.png")

![image_1780618570382.png](./image_1780618570382.png "image_1780618570382.png")

In [0]:
#conda.yaml

#channels:
#- conda-forge
#dependencies:
#- python=3.12.3
#- pip<=25.0.1
#- pip:
#  - mlflow==3.8.1
#  - cloudpickle==3.0.0
#  - pandas==2.2.3
#  - psutil==5.9.0
#  - scikit-learn==1.6.1
#name: mlflow-env


In [0]:
#python_env.yaml
#python: 3.12.3
#build_dependencies:
#- pip==25.0.1
#- setuptools==78.1.1
#- wheel==0.45.1
#dependencies:
#- -r requirements.txt




In [0]:
#requirements.txt
#mlflow==3.8.1
#cloudpickle==3.0.0
#pandas==2.2.3
#psutil==5.9.0
#scikit-learn==1.6.1

In [0]:
# página 46

## Use the Model for Inference

Now that our model is ready in model registry, we can use it for inference. In this section we will use the model for inference directly on a spark dataframe, which called **batch inference**.

### Load the Model

Loading a model from UC-based model registry is done by getting a model using **alias** and **version**.

After loading the model, we will create a **spark_udf** from the model.

In [0]:
model_name # catalog.schema.arquivo

'workspace.default.ml_model'

In [0]:
latest_model_version = client.get_model_version_by_alias(name=model_name, alias="baseline").version
model_uri = f"models:/{model_name}/{latest_model_version}" # Should be version 1
# model_uri = f"models:/{model_name}@baseline" # uri can also point to @alias
predict_func = mlflow.pyfunc.spark_udf(
    spark,
    model_uri
)

2026/06/05 00:32:27 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/06/05 00:32:27 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


### Inference

Next, we will simply use the created function for inference.

In [0]:
primary_key

'customerID'

In [0]:
features

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [0]:
# prepare test dataset
test_features_df = test_df.select(primary_key, *features)

# make prediction
prediction_df = test_features_df.withColumn("prediction", predict_func(*test_features_df.drop(primary_key).columns))

display(prediction_df.show(10)) # prediction => churn

+----------+-------------+------+--------------+------------+----------+
|customerID|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|prediction|
+----------+-------------+------+--------------+------------+----------+
|0015-UOCOJ|          1.0|   7.0|          48.2|      340.35|        No|
|0017-IUDMW|          0.0|  72.0|         116.8|     8456.75|        No|
|0019-EFAEP|          0.0|  72.0|         101.3|     7261.25|        No|
|0023-XUOPT|          0.0|  13.0|          94.1|      1215.6|       Yes|
|0030-FNXPP|          0.0|   3.0|         19.85|        57.2|        No|
|0031-PVLZI|          0.0|   4.0|         20.35|       76.35|        No|
|0032-PGELS|          0.0|   1.0|          30.5|        30.5|        No|
|0040-HALCW|          0.0|  54.0|          20.4|      1090.6|        No|
|0048-LUMLS|          0.0|  37.0|          91.2|     3247.55|        No|
|0056-EPFBG|          0.0|  20.0|          39.4|       825.4|        No|
+----------+-------------+------+--------------+---

## Batch Deployment - With Feature Store

In the previous section we trained and registered a model using Spark dataframe. In some cases, you will need to use features from a feature store for training and inference.

In this section we will demonstrate how to train and deploy a model using Feature Store.

### Create Feature Table

Let's create a feature table based on the `features_df` that we create before. Please note that we will be using **Feature Store with Unity Catalog**, which means we need to use `FeatureEngineeringClient`.

In [0]:
# página 48
#The best approach here is to add a %pip install cell before Cell 41 to install the missing databricks-feature-engineering package.



In [0]:
display(prediction_df.show(10)) # prediction => churn


+----------+-------------+------+--------------+------------+----------+
|customerID|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|prediction|
+----------+-------------+------+--------------+------------+----------+
|0015-UOCOJ|          1.0|   7.0|          48.2|      340.35|        No|
|0017-IUDMW|          0.0|  72.0|         116.8|     8456.75|        No|
|0019-EFAEP|          0.0|  72.0|         101.3|     7261.25|        No|
|0023-XUOPT|          0.0|  13.0|          94.1|      1215.6|       Yes|
|0030-FNXPP|          0.0|   3.0|         19.85|        57.2|        No|
|0031-PVLZI|          0.0|   4.0|         20.35|       76.35|        No|
|0032-PGELS|          0.0|   1.0|          30.5|        30.5|        No|
|0040-HALCW|          0.0|  54.0|          20.4|      1090.6|        No|
|0048-LUMLS|          0.0|  37.0|          91.2|     3247.55|        No|
|0056-EPFBG|          0.0|  20.0|          39.4|       825.4|        No|
+----------+-------------+------+--------------+---

In [0]:
primary_key

'customerID'

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

# prepare feature set
features_df_all = telco_df.select(primary_key, *features)

# feature table definition
fe = FeatureEngineeringClient()
# Use 'default' schema instead of 'information_schema' (user has permissions on default)
schema_name_for_features = 'default'
feature_table_name = f"{catalog_name}.{schema_name_for_features}.features"
print(feature_table_name)

#drop table if exists
try:
    fe.drop_table(name=feature_table_name)
except:
    pass

# Create feature table
fe.create_table(
    name=feature_table_name,
    df=features_df_all,
    primary_keys=[primary_key],
    description="Example feature table"
)

workspace.default.features


2026/06/05 00:51:51 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['customerID'] of table 'workspace.default.features' to NOT NULL.
2026/06/05 00:51:53 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['customerID'] on table 'workspace.default.features'.
2026/06/05 00:52:02 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.default.features'.


<FeatureTable: name='workspace.default.features', table_id='dcf2e368-5760-4028-9699-2ce58a168f82', description='Example feature table', primary_keys=['customerID'], partition_columns=[], features=['customerID', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], creation_timestamp=1780620710388, online_stores=[], notebook_producers=[], job_producers=[], table_data_sources=[], path_data_sources=[], custom_data_sources=[], timestamp_keys=[], tags={}>

In [0]:
# página 50

# Data Preparation

For this demonstration, we will utilize a fictional dataset from a Telecom Company, which includes customer information. This dataset encompasses **customer demographics**, including gender, as well as internet subscription details such as subscription plans and payment methods.

After load the dataset, we will perform simple **data cleaning and feature selection**.

In the final step, we will split the dataset to **features** and **response** sets.

## Setup Feature Lookups

In order to create a training set from the feature table, we need to define a *feature lookup*. This will be used for creating training set from the feature table.

Note that the **lookup_key** is used for matching records in feature table.

In [0]:
# Create training set based on feature lookup
from databricks.feature_engineering import FeatureLookup

fl_handle = FeatureLookup(
    table_name=feature_table_name,
    lookup_key=[primary_key]
)

training_set_spec = fe.create_training_set(
    df=response_df,
    label=response,
    feature_lookups=[fl_handle],
    exclude_columns=[primary_key]
)

# Load training dataframe based on defined feature-lookup specification
training_df = training_set_spec.load_df()

## Fit and Register a Model with UC using Feature Table

After creating the training set, **model training and registering is the same as the previous step**.

In [0]:
training_df.show(8)

+-------------+------+--------------+------------+-----+
|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|Churn|
+-------------+------+--------------+------------+-----+
|          0.0|   9.0|          65.6|       593.3|   No|
|          0.0|   9.0|          59.9|       542.4|   No|
|          0.0|   4.0|          73.9|      280.85|  Yes|
|          1.0|  13.0|          98.0|     1237.85|  Yes|
|          1.0|   3.0|          83.9|       267.4|  Yes|
|          0.0|   9.0|          69.4|      571.45|   No|
|          1.0|  71.0|         109.7|     7904.25|   No|
|          0.0|  63.0|         84.65|      5377.8|   No|
+-------------+------+--------------+------------+-----+
only showing top 8 rows


In [0]:
import warnings
from mlflow.types.utils import _infer_schema
from sklearn.tree import DecisionTreeClassifier


# Covert data to pandas dataframes
X_train_pdf2 = training_df.drop(primary_key, response).toPandas()
Y_train_pdf2 = training_df.select(response).toPandas()
clf2 = DecisionTreeClassifier(max_depth=3, random_state=42)

with mlflow.start_run(run_name="Model-Batch-Deployment-Demo-With-FS") as mlflow_run:

    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(
        log_input_examples=True,
        log_models=False,
        log_post_training_metrics=True,
        silent=True
    )

    clf2.fit(X_train_pdf2, Y_train_pdf2)

    # Infer output schema
    try:
        output_schema = _infer_schema(Y_train_pdf2)
    except Exception as e:
        warnings.warn(f"Could not infer model output schema: {e}")
        output_schema = None

    # Log using feature engineering client and push to registry
    fe.log_model(
        model=clf2,
        artifact_path="decision_tree",
        flavor=mlflow.sklearn,
        training_set=training_set_spec,
        output_schema=output_schema,
        registered_model_name=model_name
    )

# Set model alias (i.e. Champion)
client.set_registered_model_alias(model_name, "Champion", get_latest_model_version(model_name))        
    

🔗 View Logged Model at: https://dbc-0b69ef93-5247.cloud.databricks.com/ml/experiments/2770952798348874/models/m-41f98c1f67764bd2b6b46092f4df4705?o=7474657872577658
/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3285: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(
Registered model 'workspace.default.ml_model' already exists. Creating a new version of this model...
2026/06/05 01:12:02 WARNING mlflow.tracking._model_registry.fluent: Run with id b44a8fc4f4f34f3aafd4409cb445c156 has no artifacts at artifact path 'decision_tree', registering model based on models:/m-41f98c1f67764bd2b6b46092f4df4705 instead


Uploading artifacts:   0%|          | 0/15 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.default.ml_model': https://dbc-0b69ef93-5247.cloud.databricks.com/explore/data/models/workspace/default/ml_model/version/3?o=7474657872577658


## Use the Model for Inference

Inference for models that are registered with a Feature Store table is different than inference with Spark dataframe. For inference, we will use **feature engineering client's `.score_batch()` method**. This method takes **a model URI** and **dataframe with primary key info**.

**So how does the function know which feature table to use?** If you visit **Artifacts** section of registered model, you will see a **data folder** is registered with the model. Also, model file includes `data: data/feature_store` statement to define feature data.

In [0]:
champion_model_uri = f"models:/{model_name}@champion"
print(champion_model_uri)

models:/workspace.default.ml_model@champion


In [0]:
# prepare lookup dataset
lookup_df = test_df.select("customerID")

# predict in batch using lookup df
prediction_fe_df = fe.score_batch(
    model_uri=champion_model_uri,
    df=lookup_df,
    result_type='string'
)

2026/06/05 01:14:57 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.


2026/06/05 01:14:57 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [0]:
display(prediction_fe_df.show(5))

+----------+-------------+------+--------------+------------+----------+
|customerID|SeniorCitizen|tenure|MonthlyCharges|TotalCharges|prediction|
+----------+-------------+------+--------------+------------+----------+
|0015-UOCOJ|          1.0|   7.0|          48.2|      340.35|        No|
|0017-IUDMW|          0.0|  72.0|         116.8|     8456.75|        No|
|0019-EFAEP|          0.0|  72.0|         101.3|     7261.25|        No|
|0023-XUOPT|          0.0|  13.0|          94.1|      1215.6|       Yes|
|0030-FNXPP|          0.0|   3.0|         19.85|        57.2|        No|
+----------+-------------+------+--------------+------------+----------+
only showing top 5 rows


In [0]:
prediction_fe_df.count()

1407

## Performance Considerations

There are many possible (write) optimizations that Delta Lake can offer such as:
* **Partitioning:** stores data associated with different categorical values in different directories.
* **Z-Ordering:** colocates related information in the same set of files.
* **Liquid Clustering:** replaces both above-mentioned methods to simplify data layout decisions and optimize query performance.
* **Predictive Optimizations:** removes the need to manually manage maintenance operations for Delta tables on Databricks.

In this demo, we will show the last two options; liquid clustering and predictive optimization.

In [0]:
catalog_name

'workspace'

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")


DataFrame[]

In [0]:
schema_name

'default'

In [0]:
spark.sql(f"USE SCHEMA {schema_name}")

DataFrame[]

### Enable Predictive Optimization at schema level (can also be done at catalog level)

In [0]:
spark.sql(f"ALTER SCHEMA {catalog_name}.{schema_name} ENABLE PREDICTIVE OPTIMIZATION;")

DataFrame[]

### Create inference table (where batch scoring jobs would materialized) and enable liquid clustering on using CLUSTER BY

In [0]:
%sql
CREATE OR REPLACE TABLE batch_inference(
    customerID STRING
    ,Churn STRING
    ,SeniorCitizen DOUBLE
    ,tenure DOUBLE
    ,MonthlyCharges DOUBLE
    ,TotalCharges DOUBLE
    ,prediction STRING
)
CLUSTER BY (customerID, tenure)

In [0]:
(
    prediction_fe_df.write
    .mode("append")
    .option("mergeSchema", True)
    .saveAsTable(f"{catalog_name}.{schema_name}.batch_inference")
)

![image_1780622626303.png](./image_1780622626303.png "image_1780622626303.png")

## Manually optimize table

In [0]:
## Manually optimize tablez

In [0]:
%sql
ANALYZE TABLE batch_inference COMPUTE STATISTICS FOR ALL COLUMNS;
OPTIMIZE batch_inference

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1780622694333, 1780622700104, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, List(22399, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 22399, 22399, null, log, 16777216, 67108864, 4, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(196, 104, 0, 0, 0, 2430), 2, 1, 5, sizeAware, false, 0, null), null)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1780622700169, 1780622703230, 8, 0, null, List(0, 0), null, 7, 7, 0, 0, List(22399, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1487, 0, 0, 0), 15, 1, 1, null, false, 0, null), null)"


## Clean up Classroom

Run the following cell to remove lessons-specific assets created during this lesson.

In [0]:
%sql
SELECT * 
FROM batch_inference

LIMIT 50

customerID,Churn,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,prediction
0151-ONTOV,null,0.0,1.0,70.9,70.9,Yes
0238-WHBIQ,null,0.0,72.0,89.7,6339.3,No
0369-ZGOVK,null,0.0,28.0,70.4,1992.2,No
0407-BDJKB,null,0.0,60.0,95.75,5742.9,No
0455-XFASS,null,0.0,3.0,69.55,200.2,Yes
0495-RVCBF,null,0.0,1.0,79.7,79.7,Yes
0577-WHMEV,null,0.0,16.0,90.7,1374.9,Yes
0787-LHDYT,null,0.0,16.0,20.6,330.25,No
0799-DDIHE,null,0.0,15.0,46.3,639.45,No
0813-TAXXS,null,0.0,55.0,77.8,4323.35,No


## Clean up Classroom

Run the following cell to remove lessons-specific assets created during this lesson.

In [0]:
#cleanup()

## Conclusion

In this demo, we presented two main batch deployment methods using MLflow for model tracking and logging with Unity Catalog. In the first approach, we trained and registered a model without a feature table, reloading it for inference through a Spark UDF. The second method involved training a model with a feature table, registering it in the model registry, and using a look-up key for data retrieval during batch inference.